In [ ]:
%pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu pandas==2.2.3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 43.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch

print("Pandas version:", pd.__version__)
print("GPU available:", torch.cuda.is_available())

Pandas version: 2.2.3
GPU available: True


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/bedooralmareni/saudi-Digital-Concierge.git"
PROJECT_DIR = "/content/saudi-Digital-Concierge"

if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    subprocess.run(
        ["git", "clone", REPO_URL, PROJECT_DIR],
        check=True
    )

VECTOR_DIR = f"{PROJECT_DIR}/data/final/vector_store"
DB_PATH = f"{PROJECT_DIR}/data/final/knowledge_base.sqlite"

required_files = [
    f"{VECTOR_DIR}/vectors.npy",
    f"{VECTOR_DIR}/vector_meta.csv",
    f"{VECTOR_DIR}/index_info.json",
    DB_PATH
]

for path in required_files:
    assert os.path.isfile(path), f"Missing file: {path}"
    print("OK:", path)

OK: /content/saudi-Digital-Concierge/data/final/vector_store/vectors.npy
OK: /content/saudi-Digital-Concierge/data/final/vector_store/vector_meta.csv
OK: /content/saudi-Digital-Concierge/data/final/vector_store/index_info.json
OK: /content/saudi-Digital-Concierge/data/final/knowledge_base.sqlite


In [ ]:
import json
import numpy as np
import pandas as pd
import faiss

with open(f"{VECTOR_DIR}/index_info.json", encoding="utf-8") as f:
    info = json.load(f)

vectors = np.load(f"{VECTOR_DIR}/vectors.npy")
metadata = pd.read_csv(f"{VECTOR_DIR}/vector_meta.csv")

assert info["model"] == "intfloat/multilingual-e5-base"
assert info["normalized"] is True
assert len(vectors) == len(metadata)
assert vectors.shape[1] == 768

vectors = np.ascontiguousarray(vectors, dtype="float32")
assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("FAISS vectors:", index.ntotal)
print("Dimensions:", index.d)
print("SQLite exists:", os.path.isfile(DB_PATH))

FAISS vectors: 13920
Dimensions: 768
SQLite exists: True


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cpu"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model loaded.


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

assert torch.cuda.is_available(), "Please enable T4 GPU in Colab."

MODEL_ID = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("Qwen3-8B loaded successfully.")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-8B loaded successfully.


In [ ]:


import sqlite3
import numpy as np
import pandas as pd
import faiss


# Shared Arabic / English city normalization
CITY_ALIASES = {
    "الرياض": "Riyadh", "riyadh": "Riyadh",
    "جدة": "Jeddah", "جده": "Jeddah", "jeddah": "Jeddah",
    "أبها": "Abha", "ابها": "Abha", "abha": "Abha",
    "الدمام": "Dammam", "dammam": "Dammam",
    "الطائف": "Taif", "taif": "Taif",
    "المدينة": "Madinah", "المدينة المنورة": "Madinah", "madinah": "Madinah",
    "مكة": "Makkah", "مكه": "Makkah", "makkah": "Makkah",
    "العلا": "Al Ula", "al ula": "Al Ula",
}

def normalize_city(city):
    cleaned = str(city).strip()
    return CITY_ALIASES.get(cleaned.casefold(), cleaned)

# ============================================
# TOOL 1: SEMANTIC SEARCH
# Attractions, Restaurants, Tourist Places
# ============================================

def search_places(query, city, entity_type=None, top_k=5):
    city = normalize_city(city)
    type_mapping = {
        "attraction": "entertainment", "attractions": "entertainment",
        "tourist attraction": "entertainment", "tourist attractions": "entertainment",
        "restaurant": "place", "restaurants": "place",
        "cafe": "place", "cafes": "place",
    }
    if entity_type:
        mapped = [type_mapping.get(t.strip().casefold(), t.strip().casefold())
                  for t in str(entity_type).split(",") if t.strip()]
        entity_type = ",".join(dict.fromkeys(mapped))

    # ----------------------------------------
    # Filter by city
    # ----------------------------------------

    city_mask = (
        metadata["city"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        == city.strip().casefold()
    )

    mask = city_mask.copy()

    # ----------------------------------------
    # Support multiple entity types
    # Example: "attraction,restaurant"
    # ----------------------------------------

    if entity_type:

        types = [
            t.strip().casefold()
            for t in entity_type.split(",")
            if t.strip()
        ]

        type_mask = (
            metadata["entity_type"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.casefold()
            .isin(types)
        )

        mask &= type_mask

    # ----------------------------------------
    # Get matching vector indices
    # ----------------------------------------

    ids = np.flatnonzero(mask.to_numpy())

    if len(ids) == 0:
        return []

    # ----------------------------------------
    # Build temporary FAISS index
    # ----------------------------------------

    selected = np.ascontiguousarray(
        vectors[ids],
        dtype="float32"
    )

    local_index = faiss.IndexFlatIP(
        selected.shape[1]
    )

    local_index.add(selected)

    # ----------------------------------------
    # Encode user query
    # ----------------------------------------

    q = embedder.encode(
        ["query: " + query],
        normalize_embeddings=True
    ).astype("float32")

    # ----------------------------------------
    # Semantic search
    # ----------------------------------------

    scores, local_ids = local_index.search(
        q,
        min(len(ids), top_k * 5)
    )

    # ----------------------------------------
    # Remove duplicate entities
    # ----------------------------------------

    results = []
    seen = set()

    for score, local_id in zip(
        scores[0],
        local_ids[0]
    ):

        if local_id < 0:
            continue

        row = metadata.iloc[
            int(ids[local_id])
        ]

        key = row.get("canonical_id")

        if pd.isna(key):
            key = row.get(
                "entity_id",
                int(ids[local_id])
            )

        key = str(key)

        # Also deduplicate records that refer to the same named venue.
        name_key = str(row.get("text", "")).split(" — ")[0].strip().casefold()
        dedup_key = name_key or key
        if dedup_key in seen:
            continue

        seen.add(dedup_key)

        results.append({
            "score": round(float(score), 4),
            "city": str(row.get("city", "")),
            "type": str(row.get("entity_type", "")),
            "category": str(row.get("category", "")),
            "text": str(row.get("text", ""))
        })

        if len(results) >= top_k:
            break

    return results


# ============================================
# TOOL 2: EVENT SEARCH
# Filter events by city and travel dates
# ============================================

def search_events(
    city,
    start_date,
    end_date,
    limit=10
):

    city = normalize_city(city)
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    if start > end:
        raise ValueError(
            "Invalid date range"
        )

    with sqlite3.connect(DB_PATH) as conn:

        df = pd.read_sql_query(
            """
            SELECT
                event_id,
                name,
                city,
                start_date,
                end_date,
                source,
                source_url

            FROM events

            WHERE
                LOWER(TRIM(city)) = LOWER(TRIM(?))

                AND date(start_date) <= date(?)

                AND date(
                    COALESCE(end_date, start_date)
                ) >= date(?)

            ORDER BY start_date

            LIMIT ?
            """,
            conn,
            params=(
                city,
                end.strftime("%Y-%m-%d"),
                start.strftime("%Y-%m-%d"),
                limit
            )
        )

    return df.to_dict(
        orient="records"
    )


# ============================================
# TOOL 3: HOTEL SEARCH
# Filter by city, budget, rating, guests
# ============================================

def search_hotels(
    city,
    max_price_per_night=None,
    min_guest_rating=None,
    guests=None,
    limit=5
):

    city = normalize_city(city)
    conditions = [
        "LOWER(TRIM(city)) = LOWER(TRIM(?))"
    ]

    params = [city]

    # ----------------------------------------
    # Budget filter
    # ----------------------------------------

    if max_price_per_night is not None:

        conditions.append(
            "price_sar <= ?"
        )

        params.append(
            max_price_per_night
        )

    # ----------------------------------------
    # Guest rating filter
    # ----------------------------------------

    if min_guest_rating is not None:

        conditions.append(
            "guest_rating >= ?"
        )

        params.append(
            min_guest_rating
        )

    # ----------------------------------------
    # Guest capacity filter
    # ----------------------------------------

    if guests is not None:

        conditions.append(
            "max_persons >= ?"
        )

        params.append(
            guests
        )

    where_clause = " AND ".join(
        conditions
    )

    query = f"""
        SELECT
            hotel_id,
            name,
            city,
            price_sar,
            star_rating,
            guest_rating,
            max_persons,
            source,
            snapshot_date

        FROM hotels

        WHERE {where_clause}

        ORDER BY
            guest_rating DESC,
            price_sar ASC

        LIMIT ?
    """

    with sqlite3.connect(DB_PATH) as conn:

        df = pd.read_sql_query(
            query,
            conn,
            params=params + [limit]
        )

    return df.to_dict(
        orient="records"
    )


# ============================================
# READY
# ============================================

print("Knowledge Base Search Tools Ready!")
print("1. search_places")
print("2. search_events")
print("3. search_hotels")

Knowledge Base Search Tools Ready!
1. search_places
2. search_events
3. search_hotels


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": (
                 "Search the Saudi tourism knowledge base. "
    "Valid entity_type values are: place, entertainment, "
    "hotel, review. Use entertainment for tourist "
    "attractions and activities. Use place when searching "
    "for restaurants or other places, but verify the "
    "returned category and text. "
    "Some cities may not have all entity types. "
    "Omit entity_type to search all available types."



            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "max_price_per_night": {"type": "number"},
                    "min_guest_rating": {"type": "number"},
                    "guests": {"type": "integer"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_places",
            "description": (
                "Search attractions, restaurants and other "
                "tourism entities using semantic search. "
                "Use the city and user's interests."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "city": {"type": "string"},
                    "entity_type": {"type": "string"}
                },
                "required": ["query", "city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_events",
            "description": (
                "Find events recorded in the knowledge base "
                "that overlap the requested travel dates. "
                "Results do not confirm live availability."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "start_date": {"type": "string"},
                    "end_date": {"type": "string"}
                },
                "required": ["city", "start_date", "end_date"]
            }
        }
    }
]

TOOL_FUNCTIONS = {
    "search_hotels": search_hotels,
    "search_places": search_places,
    "search_events": search_events
}

print("Available tools:", list(TOOL_FUNCTIONS))

Available tools: ['search_hotels', 'search_places', 'search_events']


In [ ]:
import inspect

def execute_tool(name, arguments):
    if name not in TOOL_FUNCTIONS:
        return {"error": f"Unknown tool: {name}"}

    if not isinstance(arguments, dict):
        return {"error": "Arguments must be an object"}

    func = TOOL_FUNCTIONS[name]
    signature = inspect.signature(func)

    try:
        signature.bind(**arguments)
        result = func(**arguments)

        return json.loads(json.dumps(
            result,
            ensure_ascii=False,
            default=str
        ))

    except Exception as e:
        return {"error": str(e)}

In [ ]:


import json
import re
import time
import torch


# ============================================
# 1. SYSTEM PROMPT
# ============================================

SYSTEM_PROMPT = """
You are Saudi Digital Concierge, an AI travel planning agent
specialized in tourism within Saudi Arabia.

Your task is to respond to Saudi tourism requests using
the available knowledge base tools.

AVAILABLE TOOLS:
1. search_hotels
2. search_places
3. search_events

AGENT BEHAVIOR:
- Decide independently which tools to call.
- Use tools when information is needed.
- You may call tools multiple times.
- Answer exactly what the user requested.
- If the user asks only for hotels, recommend hotels only.
- Create a day-by-day itinerary only when requested.
- Use only information supported by retrieved tool results.
- Do not invent hotels, attractions, events, restaurants,
  prices, dates, or availability.
- Respect the user's city, dates, budget, preferences,
  and exclusions.
- Do not claim that a booking has been made.

KNOWLEDGE BASE:
- Valid entity types are:
  place, entertainment, hotel, review.
- Use entertainment for tourist attractions and activities.
- Use place for restaurants only when the category
  or text confirms that the result is a restaurant.
- Some cities may have limited coverage.
- If a search returns no results, retry with a broader
  query or fewer optional filters when appropriate.
- Never silently relax a mandatory user constraint.
- Do not recommend places outside the requested city.
- Avoid duplicate recommendations.

GROUNDING AND EVIDENCE RULES:
- Every named hotel, attraction, restaurant, and event
  recommended in the final answer MUST appear in
  the retrieved tool results.
- Do not use pretrained knowledge to invent recommendations.
- Do not invent descriptions, opening hours, prices,
  operating schedules, or availability.
- If a requested place is not retrieved, identify it
  as unverified rather than recommending it as confirmed.
- Event date ranges in the database do not prove
  current operational availability.
- Hotel prices are historical snapshots, not live quotes.
- State the snapshot date when reporting hotel prices.
- Do not present historical hotel prices as confirmed current offers.
- Do not estimate missing costs without supporting data.
- If the user asks for a budget breakdown, separate
  known costs from unknown costs.
- Do not describe hotel-only cost as total trip cost.

ITINERARY CATEGORY RULES:
- Never label an attraction, cinema, entertainment venue,
  or hotel as a restaurant or dining recommendation.
- Use the category field returned by search_places.
- Recommend restaurants only when verified by retrieved data.
- If no verified restaurants are retrieved, explain that
  restaurant information is unavailable.
- Dining slots are optional when restaurant data is missing.
- Do not fill itinerary slots with unrelated businesses.
- If a must-include place is missing from retrieved results,
  explicitly state that it could not be verified.
- Do not describe events as confirmed ongoing or available
  solely because their stored date ranges overlap the trip.
- Prioritize accuracy over producing a fully populated plan.

CONSTRAINT RULES:
- Respect strict inequalities:
  "under 300 SAR" means price < 300,
  not price <= 300.
- "At most 400 SAR" means price <= 400.
- Do not recommend options that violate mandatory constraints.
- If constraints cannot be satisfied, explain why
  and suggest possible adjustments.
- Do not silently increase the budget or lower ratings.
- Calculate hotel costs using check-out minus check-in dates,
  not the number of calendar days in the itinerary.
- If dates are not provided, do not invent exact dates.
- If an activity cost is missing, label it unknown.

LANGUAGE RULES:
- Respond in the same language as the user's original request.
- For Arabic requests, use Arabic for all explanations,
  headings, recommendations, and summaries.
- For English requests, respond in English.
- Preserve official hotel, attraction, and event names
  exactly as they appear in the knowledge base.
- Never switch the response language because tool results
  or system instructions are written in English.

FINAL RESPONSE RULES:
- Provide the actual answer, not a promise to answer.
- For itinerary requests, provide a day-by-day plan.
- For hotel-only requests, provide the requested hotel list.
- Include only supported recommendations.
- Clearly identify missing information.
- Do not claim that an unavailable option is available.
- Keep the response organized and practical.
"""


# ============================================
# 2. DETECT USER LANGUAGE
# ============================================

def detect_user_language(text):

    arabic_chars = re.findall(
        r"[\u0600-\u06FF]",
        text
    )

    english_chars = re.findall(
        r"[A-Za-z]",
        text
    )

    if len(arabic_chars) > len(english_chars):
        return "Arabic"

    return "English"


# ============================================
# 3. CHECK FINAL ANSWER LANGUAGE
# ============================================

def is_wrong_language(answer, expected_language):

    clean_answer = re.sub(
        r"<tool_call>.*?</tool_call>",
        "",
        answer,
        flags=re.DOTALL
    )

    # Remove code blocks and URLs
    clean_answer = re.sub(
        r"```.*?```",
        "",
        clean_answer,
        flags=re.DOTALL
    )

    clean_answer = re.sub(
        r"https?://\S+",
        "",
        clean_answer
    )

    arabic_count = len(
        re.findall(
            r"[\u0600-\u06FF]",
            clean_answer
        )
    )

    english_count = len(
        re.findall(
            r"[A-Za-z]",
            clean_answer
        )
    )

    if expected_language == "Arabic":
        return arabic_count < english_count * 0.5

    return english_count < arabic_count * 0.5


# ============================================
# 4. PARSE QWEN TOOL CALLS
# ============================================

def parse_tool_calls(reply):

    calls = []

    pattern = (
        r"<tool_call>\s*"
        r"(\{.*?\})"
        r"\s*</tool_call>"
    )

    matches = re.findall(
        pattern,
        reply,
        flags=re.DOTALL
    )

    for match in matches:

        try:

            call = json.loads(match)

            name = call.get("name")
            arguments = call.get("arguments", {})

            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            if (
                isinstance(name, str)
                and isinstance(arguments, dict)
            ):
                calls.append({
                    "name": name,
                    "arguments": arguments
                })

        except (json.JSONDecodeError, TypeError):
            continue

    return calls


# ============================================
# 5. CHECK ANSWER COMPLETENESS
# ============================================

def is_incomplete_answer(answer, user_request=""):

    text = answer.lower().strip()

    incomplete_phrases = [
        "i will proceed",
        "i will create",
        "i will now",
        "let me create",
        "i can create",
        "سأقوم بإعداد",
        "سوف أقوم بإعداد",
        "سأبدأ بإعداد"
    ]

    if any(
        phrase in text
        for phrase in incomplete_phrases
    ):
        return True

    if len(text) < 60:
        return True

    request = user_request.lower()

    # Only require day headings when an itinerary
    # is requested for a specific number of days.
    itinerary_requested = any(
        word in request
        for word in [
            "trip",
            "itinerary",
            "travel plan",
            "رحلة",
            "جدول",
            "خطط لي"
        ]
    )

    if not itinerary_requested:
        return False

    three_days = any(
        phrase in request
        for phrase in [
            "3-day",
            "3 day",
            "three-day",
            "3 أيام",
            "3 ايام",
            "ثلاثة أيام",
            "ثلاث ايام"
        ]
    )

    two_days = any(
        phrase in request
        for phrase in [
            "2-day",
            "2 day",
            "two-day",
            "يومين",
            "يومَين"
        ]
    )

    has_day_1 = bool(re.search(
        r"\bday\s*1\b|اليوم\s*الأول|اليوم\s*1",
        text
    ))

    has_day_2 = bool(re.search(
        r"\bday\s*2\b|اليوم\s*الثاني|اليوم\s*2",
        text
    ))

    has_day_3 = bool(re.search(
        r"\bday\s*3\b|اليوم\s*الثالث|اليوم\s*3",
        text
    ))

    # An infeasible request may correctly result
    # in an explanation rather than an itinerary.
    infeasible_phrases = [
        "غير ممكن",
        "لا يمكن",
        "لا توجد خيارات",
        "لا توجد نتائج",
        "cannot satisfy",
        "not feasible",
        "no matching",
        "no available options"
    ]

    if any(
        phrase in text
        for phrase in infeasible_phrases
    ):
        return False

    if three_days:
        return not (
            has_day_1
            and has_day_2
            and has_day_3
        )

    if two_days:
        return not (
            has_day_1
            and has_day_2
        )

    return False


# ============================================
# 6. RUN AGENT
# ============================================

def run_agent(
    user_request,
    max_steps=10,
    max_retries=2
):

    start_time = time.time()

    user_language = detect_user_language(
        user_request
    )

    if user_language == "Arabic":

        language_instruction = """
تعليمات مهمة جدًا:
يجب أن تكون الإجابة النهائية باللغة العربية.
استخدم العربية في العناوين والشرح والتوصيات.
احتفظ بأسماء الفنادق والأماكن الرسمية كما هي.
لا تكتب الإجابة النهائية باللغة الإنجليزية.
"""

    else:

        language_instruction = """
IMPORTANT:
Your final answer must be in English.
Preserve official hotel and place names.
"""

    messages = [
        {
            "role": "system",
            "content": (
                SYSTEM_PROMPT
                + "\n\n"
                + language_instruction
            )
        },
        {
            "role": "user",
            "content": user_request
        }
    ]

    trace = []
    previous_calls = set()

    retry_count = 0
    last_answer = ""

    for step in range(1, max_steps + 1):

        print(f"\n=== AGENT STEP {step} ===")

        # ------------------------------------
        # Prepare Qwen input
        # ------------------------------------

        inputs = tokenizer.apply_chat_template(
            messages,
            tools=TOOLS,
            add_generation_prompt=True,
            enable_thinking=False,
            return_tensors="pt",
            return_dict=True
        ).to(model.device)

        # ------------------------------------
        # Generate model response
        # ------------------------------------

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=1200,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        input_length = inputs[
            "input_ids"
        ].shape[-1]

        reply = tokenizer.decode(
            outputs[0][input_length:],
            skip_special_tokens=True
        ).strip()

        last_answer = reply

        # ------------------------------------
        # Parse tool calls
        # ------------------------------------

        calls = parse_tool_calls(reply)

        # ====================================
        # CASE A — FINAL RESPONSE
        # ====================================

        if not calls:

            if "<tool_call>" in reply:

                print(
                    "Malformed tool call detected."
                )

                return {
                    "status": "incomplete_tool_call",
                    "answer": reply,
                    "trace": trace,
                    "elapsed_seconds": round(
                        time.time() - start_time, 2
                    )
                }

            incomplete = is_incomplete_answer(
                reply,
                user_request
            )

            wrong_language = is_wrong_language(
                reply,
                user_language
            )

            if (
                (incomplete or wrong_language)
                and retry_count < max_retries
            ):

                retry_count += 1

                print(
                    "Answer needs correction. "
                    f"Retry {retry_count}/{max_retries}"
                )

                if wrong_language:
                    print(
                        "Reason: Wrong response language."
                    )

                if incomplete:
                    print(
                        "Reason: Incomplete response."
                    )

                messages.append({
                    "role": "assistant",
                    "content": reply
                })

                if user_language == "Arabic":

                    correction = """
إجابتك السابقة تحتاج إلى تصحيح.

أجب الآن باللغة العربية فقط.
استخدم أسماء الفنادق والأماكن كما وردت
في نتائج الأدوات.

إذا طلب المستخدم فنادق فقط،
اعرض الفنادق المطلوبة دون إنشاء رحلة.

إذا طلب خطة رحلة،
قدم جدول الأيام المطلوب.

لا تخترع معلومات غير موجودة
في نتائج الأدوات.

لا تكتفِ بوعد المستخدم بإعداد الإجابة.
قدم الإجابة الفعلية الآن.
"""

                else:

                    correction = """
Your previous answer needs correction.

Respond entirely in English.
Use only evidence retrieved from tools.

If the user requested hotels only,
provide the hotel recommendations.

If the user requested an itinerary,
provide the actual day-by-day itinerary.

Do not invent missing information.
Do not merely promise to answer.
"""

                messages.append({
                    "role": "user",
                    "content": correction
                })

                continue

            if incomplete or wrong_language:

                print(
                    "Agent stopped with an "
                    "incomplete or wrong-language answer."
                )

                return {
                    "status": "incomplete",
                    "answer": reply,
                    "trace": trace,
                    "elapsed_seconds": round(
                        time.time() - start_time, 2
                    )
                }

            print("Agent finished.")

            return {
                "status": "completed",
                "answer": reply,
                "trace": trace,
                "elapsed_seconds": round(
                    time.time() - start_time, 2
                )
            }

        # ====================================
        # CASE B — TOOL CALLS
        # ====================================

        messages.append({
            "role": "assistant",
            "content": reply
        })

        for call in calls:

            name = call["name"]
            arguments = call["arguments"]

            print(f"\nTool selected: {name}")
            print(f"Arguments: {arguments}")

            call_key = json.dumps(
                {
                    "name": name,
                    "arguments": arguments
                },
                sort_keys=True,
                ensure_ascii=False
            )

            if call_key in previous_calls:

                result = {
                    "warning": (
                        "This exact tool call was already "
                        "executed. Use the previous result "
                        "or try different arguments."
                    )
                }

                print(
                    "Repeated tool call detected."
                )

            else:

                previous_calls.add(call_key)

                try:

                    result = execute_tool(
                        name,
                        arguments
                    )

                except Exception as e:

                    result = {
                        "error": str(e)
                    }

            if isinstance(result, list):

                print(
                    "Records returned:",
                    len(result)
                )

            else:

                print(
                    "Tool response:",
                    str(result)[:300]
                )

            trace.append({
                "step": step,
                "tool": name,
                "arguments": arguments,
                "result": result
            })

            messages.append({
                "role": "tool",
                "name": name,
                "content": json.dumps(
                    result,
                    ensure_ascii=False,
                    default=str
                )
            })

    # ========================================
    # MAXIMUM STEPS
    # ========================================

    print(
        "\nMaximum agent steps reached."
    )

    return {
        "status": "max_steps_reached",
        "answer": last_answer,
        "trace": trace,
        "elapsed_seconds": round(
            time.time() - start_time, 2
        )
    }


print("Qwen3 Single-Agent Loop Ready!")
print("Arabic/English language detection enabled.")
print("Answer completeness checking enabled.")
print("Knowledge-base grounding instructions enabled.")

Qwen3 Single-Agent Loop Ready!
Arabic/English language detection enabled.
Answer completeness checking enabled.
Knowledge-base grounding instructions enabled.


In [ ]:

#  FIVE BASELINE TEST SCENARIOS


import time

SCENARIOS = {

    # ========================================
    # S1 — Simple Hotel Recommendation
    # ========================================

    "S1": """
أبغى فندق في الرياض لشخصين،
سعر الليلة ما يتجاوز 400 ريال،
وتقييم النزلاء يكون 8 من 10 أو أعلى.
اقترح لي أفضل 3 خيارات.
""",

    # ========================================
    # S2 — Personalized Travel Plan
    # ========================================

    "S2": """
Plan a 3-day trip to Jeddah for two people.
I enjoy entertainment and family-friendly activities.
Recommend a hotel under 300 SAR per night
with a guest rating of at least 8/10,
along with suitable places to visit.
""",

    # ========================================
    # S3 — Nature Trip with Constraints
    # ========================================

    "S3": """
خطط لي رحلة يومين إلى أبها لشخصين.
نحب الطبيعة والأنشطة الخارجية،
ونبغى سكن بسعر أقل من 350 ريال لليلة
وتقييمه 7 من 10 أو أعلى.
رتب لنا جدولًا مناسبًا،
ولا تقترح أماكن خارج أبها.
""",

    # ========================================
    # S4 — Multi-Constraint Planning
    # ========================================

    "S4": """
Plan a 3-day trip to Riyadh for two people
from October 15 to 17, 2026.

Our total budget is 1,500 SAR,
including accommodation and activities.

We want cultural attractions, entertainment,
and restaurants rated at least 4/5.

Avoid shopping malls,
and include only events that overlap
our travel dates.

Provide a daily itinerary and
an estimated cost breakdown.

Clearly identify any costs that are
unavailable in the dataset.
""",

    # ========================================
    # S5 — Conflicting Constraints
    # ========================================

    "S5": """
أبغى رحلة إلى جدة لمدة 3 أيام لشخصين،
بميزانية إجمالية 300 ريال تشمل السكن والأنشطة.

لازم يكون الفندق تقييمه 9 من 10 أو أعلى
وسعر الليلة أقل من 50 ريال.

وأبغى مطعمين تقييمهم 4.5 من 5 أو أعلى.

لا تتجاوز الميزانية أو تتجاهل أي شرط،
وإذا الطلب غير ممكن وضح لي السبب
واقترح تعديلات مناسبة.
"""
}




In [ ]:
# ============================================
# RUN ALL FIVE BASELINE SCENARIOS
# ============================================

import io
import contextlib

ALL_RESULTS = {}

for scenario_id, user_request in SCENARIOS.items():

    # Hide internal agent logs and tool calls
    with contextlib.redirect_stdout(io.StringIO()):
        result = run_agent(
            user_request=user_request,
            max_steps=10,
            max_retries=2
        )

    # Save results for later evaluation
    ALL_RESULTS[scenario_id] = {
        "request": user_request.strip(),
        "answer": result["answer"],
        "status": result["status"],
        "trace": result["trace"],
        "elapsed_seconds": result["elapsed_seconds"]
    }

    # Display only scenario and Qwen answer
    print("\n" + "-" * 65)
    print(f"SCENARIO {scenario_id}")
    print("-" * 65)

    print("\nUSER REQUEST:\n")
    print(user_request.strip())

    print("\nQWEN ANSWER:\n")
    print(result["answer"])

    print("\n" + "-" * 65)


-----------------------------------------------------------------
SCENARIO S1
-----------------------------------------------------------------

USER REQUEST:

أبغى فندق في الرياض لشخصين،
سعر الليلة ما يتجاوز 400 ريال،
وتقييم النزلاء يكون 8 من 10 أو أعلى.
اقترح لي أفضل 3 خيارات.

QWEN ANSWER:

أفضل 3 خيارات للفندق في الرياض، مع سعر لا يتجاوز 400 ريال وتقييم النزلاء 8 من 10 أو أعلى:

1. **Ewaa Express Hotel - Gaber**  
   - السعر: 375 ريال لليلة  
   - التقييم: 9.1 من 10  
   - عدد الأشخاص المسموح به: 2  

2. **ضيافة الفيحاء Dayafat Alfayha'a**  
   - السعر: 200 ريال لليلة  
   - التقييم: 8.9 من 10  
   - عدد الأشخاص المسموح به: 2  

3. **Shaza Riyadh**  
   - السعر: 400 ريال لليلة  
   - التقييم: 8.8 من 10  
   - عدد الأشخاص المسموح به: 2  

كل هذه الخيارات تلبي متطلباتك.

-----------------------------------------------------------------

-----------------------------------------------------------------
SCENARIO S2
-----------------------------------------------------------------

USE